# AmsiNet - Step 26: ablations with the EfficientNet-B0 word agent (AmsiNet-E)
Repeats the fusion ablations of Table 9 (which used the EfficientViT-B0 word agent) for **AmsiNet-E**, using the saved probabilities of Steps 3, 16 and 25. **No training, no GPU: run it on a CPU runtime with *Run all* (a few minutes).** Everything is fitted on validation and evaluated once on test, five seeds.
Paste back the outputs of sections 5 and 6.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['flat_effb0'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

## 3. Costs

In [ ]:
try:
    import torch
except ImportError: torch = None
GFW = {'word': float(eff['word']), 'word_moe': float(eff['word_moe']), 'flatvit': FLAT}
TAG = {'word': 'word', 'word_moe': 'word_moe', 'flatvit': 'flatvit'}
CAND = {'flat_vits': ('vit_small_patch16_224.augreg_in21k', 9.2), 'flat_effb0': ('efficientnet_b0.ra_in1k', 0.8)}   # fallback GFLOPs if the counter is unavailable
for name, (mn, fb) in CAND.items():
    if all(os.path.exists(f'{ROOT}/agents/{name}_seed{s}_{sp}_probs.npy') for s in CFG['SEEDS'] for sp in ('val', 'test')):
        try:
            import timm
            from torch.utils.flop_counter import FlopCounterMode
            m = timm.create_model(mn, pretrained=False, num_classes=C).eval(); x = torch.zeros(1, 3, 224, 224)
            with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x)
            gf = fc.get_total_flops() / 1e9
        except Exception as e: print('FLOP counter failed for', name, e, '-> using', fb); gf = fb
        GFW[name] = gf; TAG[name] = name; print(f'{name}: {gf:.2f} GFLOPs per word')
    else: print('not available yet:', name)
VARS = [v for v in ['word_moe', 'word', 'flatvit', 'flat_vits', 'flat_effb0'] if v in GFW]; print('variants:', VARS)

## 4. Ablation helpers

In [ ]:
from sklearn.linear_model import LogisticRegression
E = 'flat_effb0'; assert E in GFW, 'flat_effb0 files (Step 16) are missing on Drive'
TRAIN_PRIOR = np.bincount([SIDX[s] for s in META[META.split == 'train'].script], minlength=C).astype(float); TRAIN_PRIOR /= TRAIN_PRIOR.sum()
RHOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
def sc(T, wl=0, wb=0, al=0, ab=0): return T['word'] + wl * T['line_ctx'] + wb * T['block_ctx'] + al * T['line_agent'] + ab * T['block_agent']
def dis_of(g, ng, pred):
    M = np.zeros((ng, C)); np.add.at(M, (g, pred), 1); return (1 - M.max(1) / M.sum(1))[g]
def casc(T, tau, W, rho=1.0, use_line=True, use_block=True):
    s = sc(T, W['line_ctx'], W['block_ctx']); u = unc(s); li = np.zeros(T['nl'], bool); bi = np.zeros(T['nb'], bool)
    if use_line:
        ok = dis_of(T['lg'], T['nl'], s.argmax(1)) <= rho + 1e-9; np.logical_or.at(li, T['lg'], (u > tau) & ok); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    if use_block:
        u = unc(s); ok = dis_of(T['bg'], T['nb'], s.argmax(1)) <= rho + 1e-9; np.logical_or.at(bi, T['bg'], (u > tau) & ok); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def fit_W(TV, rho=1.0):
    f = lambda W: metr(TV['y'], casc(TV, 0, W, rho)[0].argmax(1))['f1']; W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: f({**W, k: g}))
    return W, f(W)
def pick_tau(TV, W, rho, GF):
    cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(casc(TV, t, W, rho)) for t in TAUS}
    f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
    ok = [t for t in TAUS if cv[t][0] >= target]; return min(ok, key=lambda t: cv[t][1]) if ok else 0.0
def mixed_mask(T):
    first = np.full(T['nl'], -1); mixed = np.zeros(T['nl'], bool)
    for g, yy in zip(T['lg'], T['y']):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    return mixed[T['lg']]
def feats(T): return np.concatenate([T['word'], T['line_ctx'], T['block_ctx'], T['line_agent'], T['block_agent']], 1)
def ablate(seed):
    GF = dict(word=GFW[E], line=float(eff['line']), block=float(eff['block']))
    wtag, ltag, btag = f'{E}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag); y = TT['y']; mw = mixed_mask(TT)
    Z_l, O_l, Z_b, O_b = np.zeros(TT['nl'], bool), np.ones(TT['nl'], bool), np.zeros(TT['nb'], bool), np.ones(TT['nb'], bool)
    W, _ = fit_W(TV); tau = pick_tau(TV, W, 1.0, GF); U = {k: 1.0 for k in KEYS}; tau_u = pick_tau(TV, U, 1.0, GF)
    fits = {r: fit_W(TV, r) for r in RHOS}; rho = max(RHOS, key=lambda r: (round(fits[r][1], 6), r)); Wg = fits[rho][0]; tau_g = pick_tau(TV, Wg, rho, GF)
    S_ = {}
    S_['word alone'] = (sc(TT), Z_l, Z_b)
    S_['+ line context only'] = (sc(TT, W['line_ctx']), Z_l, Z_b)
    S_['+ block context only'] = (sc(TT, 0, W['block_ctx']), Z_l, Z_b)
    S_['free context, fitted weights'] = (sc(TT, W['line_ctx'], W['block_ctx']), Z_l, Z_b)
    S_['free context, unit weights'] = (sc(TT, 1, 1), Z_l, Z_b)
    S_['line agent everywhere, no context'] = (sc(TT, 0, 0, W['line_agent']), O_l, Z_b)
    S_['block agent everywhere, no context'] = (sc(TT, 0, 0, 0, W['block_agent']), Z_l, O_b)
    S_['both agents everywhere, no context'] = (sc(TT, 0, 0, W['line_agent'], W['block_agent']), O_l, O_b)
    S_['all agents everywhere (full)'] = casc(TT, 0, W)
    S_['cascade (AmsiNet-E)'] = casc(TT, tau, W)
    S_['cascade, line agent removed'] = casc(TT, tau, W, use_line=False)
    S_['cascade, block agent removed'] = casc(TT, tau, W, use_block=False)
    S_['cascade, unit weights'] = casc(TT, tau_u, U)
    S_['cascade + disagreement gate'] = casc(TT, tau_g, Wg, rho)
    s_all = casc(TT, 0, W)[0].argmax(1); orc = np.where(mw, TT['word'].argmax(1), s_all)
    lr = LogisticRegression(max_iter=5000, C=1.0).fit(feats(TV), TV['y']); st = lr.predict_proba(feats(TT)).argmax(1)
    pri = {a: metr(TV['y'], (casc(TV, 0, W)[0] - a * np.log(TRAIN_PRIOR)).argmax(1))['f1'] for a in [0, .25, .5, .75, 1]}; a_star = max(pri, key=lambda a: (round(pri[a], 6), -a))
    out = dict(seed=seed, tau=tau, tau_unit=tau_u, rho=rho, tau_gate=tau_g, prior_a=a_star, **{f'W_{k}': W[k] for k in KEYS})
    for name, (s, li, bi) in S_.items():
        m = metr(y, s.argmax(1)); out[name + '|acc'], out[name + '|f1'], out[name + '|gf'] = m['acc'], m['f1'], cost(TT, li, bi, GF)
        out[name + '|acc_mixed'] = 100 * float((s.argmax(1)[mw] == y[mw]).mean())
    for name, p in (('oracle gate (bound only)', orc), ('logistic stacking on validation', st), ('prior correction (full)', (casc(TT, 0, W)[0] - a_star * np.log(TRAIN_PRIOR)).argmax(1))):
        m = metr(y, p); out[name + '|acc'], out[name + '|f1'] = m['acc'], m['f1']; out[name + '|gf'] = np.nan
    return out

## 5. Run the five seeds

In [ ]:
rows = []
for seed in CFG['SEEDS']:
    o = ablate(seed); rows.append(o); print(seed, 'tau*', o['tau'], 'rho*', o['rho'], 'prior a*', o['prior_a'], '| cascade %.2f / %.2f (%.1f GF)' % (o['cascade (AmsiNet-E)|acc'], o['cascade (AmsiNet-E)|f1'], o['cascade (AmsiNet-E)|gf']))
AB = pd.DataFrame(rows); AB.to_csv(f'{ROOT}/results/step26_ablations_effb0.csv', index=False)

## 6. Summary (mean +/- 95% t-interval over seeds; last columns: difference to the cascade, paired over seeds)  -  paste this back

In [ ]:
def mci(x):
    x = np.asarray(x, float); n = len(x); return x.mean(), (stats.t.ppf(.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan'))
names = [c[:-4] for c in AB.columns if c.endswith('|acc')]; ref = 'cascade (AmsiNet-E)'; R = []
for n in names:
    a, f = AB[n + '|acc'], AB[n + '|f1']; da, df_ = mci(a.values - AB[ref + '|acc'].values), mci(f.values - AB[ref + '|f1'].values)
    R.append(dict(system=n, accuracy='%.2f +/- %.2f' % mci(a), macroF1='%.2f +/- %.2f' % mci(f), GFLOPs_per_image=('%.1f' % AB[n + '|gf'].mean()) if AB[n + '|gf'].notna().all() else '-',
                  d_acc='%+.2f +/- %.2f' % da, d_F1='%+.2f +/- %.2f' % df_))
display(pd.DataFrame(R)); pd.DataFrame(R).to_csv(f'{ROOT}/results/step26_summary.csv', index=False)
print('fitted weights (mean over seeds):', {k: round(AB[f'W_{k}'].mean(), 2) for k in KEYS}, '| tau*:', AB.tau.tolist(), '| gate rho*:', AB.rho.tolist(), '| prior strength:', AB.prior_a.tolist())
print('accuracy on words in mixed-script lines (mean):', {n: round(AB[n + '|acc_mixed'].mean(), 2) for n in names if (n + '|acc_mixed') in AB.columns})